# Spark-X2.6-4B-En4B-A1B (`v0.3` / 72L) — Frontier Benchmark Suite (`SWE-bench Pro`, `SWE-bench Verified`, `GPQA Diamond`, `IFEval`, `IFBench`, `HLE`, `HumanEval`, `MATH-500`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaiTian6641/Spark-X2.6-Recursion/blob/main/SparkX_Loop_v03_Frontier_Benchmarks_Colab.ipynb)

This notebook evaluates **`TassP0lar/Spark-X2.6-4B-En4B-A1B`** (`8.23B` Total / `5.23B` Active, 72 Effective Layers, 8-Anchor 32-Expert `Top-7` `MoE-Engram`) across 8 standard frontier LLM benchmarks loaded directly from Hugging Face:
1. **Cell 1**: Environment, Dependencies & Official `allenai/IFBench` Verifier Setup
2. **Cell 2**: One-Call 72-Layer Model Load (`TassP0lar/Spark-X2.6-4B-En4B-A1B`) & Sample-Cap Configuration
3. **Cell 3**: **`IFEval` (`google/IFEval`) & `IFBench` (`allenai/IFBench_test`)** — Strict/Loose Prompt & Instruction Following (`25` classic + `58` OOD constraint verifiers)
4. **Cell 4**: **`GPQA Diamond` (`Idavidrein/gpqa` / mirrors) & `HLE — Humanity's Last Exam` (`cais/hle`)** — PhD Science & Frontier Academic Exam Suite
5. **Cell 5**: **`SWE-bench Pro` (`ScaleAI/SWE-bench_Pro`) & `SWE-bench Verified` (`princeton-nlp/SWE-bench_Verified`)** — Long-Horizon Software Engineering Patch Synthesis + Official `predictions.jsonl` Export
6. **Cell 6**: **`HumanEval` (`openai/openai_humaneval`) & `MATH-500` (`HuggingFaceH4/MATH-500`)** — Executable Python `Pass@1` & Competition Math Levels 1–5
7. **Cell 7**: **Unified Leaderboard Scorecard & JSON Report Export**

In [ ]:
# Cell 1: Environment, Dependencies & Official AllenAI IFBench Repository Setup
import os, subprocess, sys
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/BaiTian6641/Spark-X2.6-Recursion.git"
REPO_DIR = Path("/content/Spark-X2.6-Recursion")

if Path("/content").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    # Clone official AllenAI IFBench verifier repository for 83-rule OOD instruction verification
    if not Path("/content/IFBench").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/allenai/IFBench.git", "/content/IFBench"], check=False)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "datasets>=3.0.0", "transformers>=4.48.0", "accelerate>=1.2.0", "safetensors>=0.4.5",
    "huggingface_hub>=0.27.0", "sympy", "langdetect", "immutabledict", "nltk", " emoji"
], check=False)

import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")

In [ ]:
# Cell 2: Load Trained Spark-X2.6-4B-En4B-A1B (72L Full Loop + 32-Expert Top-7 MoE-Engram)
import importlib, os, subprocess, torch
subprocess.run(["git", "pull", "--ff-only"], check=False)
import sparkx_loop_v03_core, load_inference_v03, sparkx_loop_v03_benchmarks
importlib.reload(sparkx_loop_v03_core)
importlib.reload(load_inference_v03)
importlib.reload(sparkx_loop_v03_benchmarks)
from load_inference_v03 import load_sparkx_loop_v03

HF_REPO_ID = "TassP0lar/Spark-X2.6-4B-En4B-A1B"
HF_TOKEN = os.environ.get("HF_TOKEN") or None
if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if "model" not in globals() or model is None:
    model, tokenizer = load_sparkx_loop_v03(
        repo_id=HF_REPO_ID,
        device="cuda" if torch.cuda.is_available() else "cpu",
        dtype=torch.bfloat16,
        offload_tier1b_cpu=False,
        token=HF_TOKEN,
    )

# Configure 72L Full Loop + MoE-Engram (optimal decode_engram_scale = 0.25 from Cell 3 ablation)
model.set_loop(start=4, end=28, passes=2, use_mhc=True, use_engram=True, enable_p3=True, p3_start=12, p3_end=24)
for _m in model.all_engram_modules():
    _m.decode_engram_scale = 0.25

# Adjust sample caps per benchmark (set to None to evaluate the full dataset split!)
BENCHMARK_LIMITS = {
    "IFEval": 25,              # Full dataset = 541 prompts (`None` for all 541)
    "IFBench": 25,             # Full dataset = 300 OOD prompts (`None` for all)
    "GPQA_Diamond": 25,        # Full dataset = 198 PhD science questions (`None` for all 198)
    "HLE": 15,                 # Humanity's Last Exam (`cais/hle`, text-only subset)
    "SWE_bench_Pro": 10,       # ScaleAI/SWE-bench_Pro (`v2` 642 tasks / `hard` 51 tasks)
    "SWE_bench_Verified": 10,  # princeton-nlp/SWE-bench_Verified (500 human-verified issues)
    "HumanEval": 25,           # Full dataset = 164 Python problems (`None` for all 164)
    "MATH_500": 25,            # Full dataset = 500 competition math problems (`None` for all 500)
}
ALL_BENCHMARK_RESULTS = {}
print("Configured Benchmark Sample Caps:", BENCHMARK_LIMITS)

In [ ]:
# Cell 3: Run IFEval (`google/IFEval`) & IFBench (`allenai/IFBench_test`)
import importlib
import sparkx_loop_v03_benchmarks
importlib.reload(sparkx_loop_v03_benchmarks)
from sparkx_loop_v03_benchmarks import run_ifeval_and_ifbench_suite

if_results = run_ifeval_and_ifbench_suite(
    model,
    tokenizer,
    max_samples_ifeval=BENCHMARK_LIMITS["IFEval"],
    max_samples_ifbench=BENCHMARK_LIMITS["IFBench"],
    token=HF_TOKEN,
)
ALL_BENCHMARK_RESULTS.update(if_results)

In [ ]:
# Cell 4: Run GPQA Diamond (`Idavidrein/gpqa`) & Humanity's Last Exam (`cais/hle`)
import importlib
import sparkx_loop_v03_benchmarks
importlib.reload(sparkx_loop_v03_benchmarks)
from sparkx_loop_v03_benchmarks import run_gpqa_diamond_benchmark, run_hle_benchmark

gpqa_res = run_gpqa_diamond_benchmark(
    model,
    tokenizer,
    max_samples=BENCHMARK_LIMITS["GPQA_Diamond"],
    token=HF_TOKEN,
)
ALL_BENCHMARK_RESULTS["GPQA_Diamond"] = gpqa_res

hle_res = run_hle_benchmark(
    model,
    tokenizer,
    max_samples=BENCHMARK_LIMITS["HLE"],
    token=HF_TOKEN,
)
ALL_BENCHMARK_RESULTS["HLE"] = hle_res

In [ ]:
# Cell 5: Run SWE-bench Pro (`ScaleAI/SWE-bench_Pro`) & SWE-bench Verified (`princeton-nlp/SWE-bench_Verified`)
import importlib
import sparkx_loop_v03_benchmarks
importlib.reload(sparkx_loop_v03_benchmarks)
from sparkx_loop_v03_benchmarks import run_swe_bench_suite

swe_res = run_swe_bench_suite(
    model,
    tokenizer,
    max_samples_pro=BENCHMARK_LIMITS["SWE_bench_Pro"],
    max_samples_verified=BENCHMARK_LIMITS["SWE_bench_Verified"],
    swebench_pro_config="default",  # Use "default" (v2, 642 tasks) or "hard" (HARD-51 subset)
    token=HF_TOKEN,
)
ALL_BENCHMARK_RESULTS.update(swe_res)

In [ ]:
# Cell 6: Run Executable HumanEval (`openai/openai_humaneval`) & Competition MATH-500 (`HuggingFaceH4/MATH-500`)
import importlib
import sparkx_loop_v03_benchmarks
importlib.reload(sparkx_loop_v03_benchmarks)
from sparkx_loop_v03_benchmarks import run_humaneval_benchmark, run_math500_benchmark

humaneval_res = run_humaneval_benchmark(
    model,
    tokenizer,
    max_samples=BENCHMARK_LIMITS["HumanEval"],
)
ALL_BENCHMARK_RESULTS["HumanEval"] = humaneval_res

math500_res = run_math500_benchmark(
    model,
    tokenizer,
    max_samples=BENCHMARK_LIMITS["MATH_500"],
)
ALL_BENCHMARK_RESULTS["MATH_500"] = math500_res

In [ ]:
# Cell 7: Unified Frontier Benchmark Scorecard & JSON Report Export
import json
from pathlib import Path

out_dir = Path("/content/sparkx_v03_benchmark_results")
out_dir.mkdir(parents=True, exist_ok=True)
report_path = out_dir / "sparkx_v03_frontier_benchmark_report.json"
report_path.write_text(json.dumps(ALL_BENCHMARK_RESULTS, indent=2), encoding="utf-8")

print("=" * 104)
print("SPARK-X2.6-4B-En4B-A1B (72L FULL LOOP + 32-EXPERT TOP-7 MoE-ENGRAM) — FRONTIER BENCHMARK SCORECARD")
print("=" * 104)
for b_name, b_metrics in ALL_BENCHMARK_RESULTS.items():
    print(f"  * {b_name:20s} : {b_metrics}")
print("=" * 104)
print(f"Saved full JSON report to: {report_path}")

In [ ]:
# ==============================================================================
# Cell 8: Phase 4 Post-Training + Supervised Instructed Self-Recursive Self-Improvement (`SI-SRSI`)
#   1. Phase 4 Multi-Task Engram & mHC Consolidation
#   2. SI-SRSI Dynamic Knowledge Engram Formation (Supervised by OpenCode Teacher):
#      - Domain 1 (`anti_slop`, Experts 22..29): Human-natural composition, reports & anti-Claude-slop unlikelihood
#      - Domain 2 (`web_ux_sec`, Experts 12..21): Full-stack UX/WCAG 2.2 policy + defensive AppSec (CSP/CSRF/SQLi/XSS)
#      - Domain 3 (`stem_reason`, Experts 0..11): First-principles STEM derivations & symbolic reasoning
# ==============================================================================
import os
import importlib
import sparkx_loop_v03_benchmarks as bench
importlib.reload(bench)

RUN_PHASE4_CONSOLIDATION = True
RUN_SI_SRSI_ENGRAM_LOOP  = True
PUSH_CONSOLIDATED_TO_HUB = True

if RUN_PHASE4_CONSOLIDATION:
    phase4_report = bench.run_phase4_engram_consolidation(
        model=model,
        tokenizer=tokenizer,
        num_steps=120,
        lr_engram=1.5e-4,
        lr_gates=5.0e-4,
        lr_mhc=3.0e-5,
        train_mhc=True,
        max_seq_len=768,
        eval_every=20,
        save_dir='/content/sparkx_v03_phase4_consolidated',
        push_to_hub=PUSH_CONSOLIDATED_TO_HUB and bool(HF_TOKEN),
        hub_repo_id=HF_REPO_ID,
        token=HF_TOKEN,
    )

if RUN_SI_SRSI_ENGRAM_LOOP:
    si_srsi_report = bench.run_si_srsi_engram_loop(
        model=model,
        tokenizer=tokenizer,
        num_rounds=2,
        consolidation_steps_per_prompt=6,
        lr_engram=1.5e-4,
        lr_gates=5.0e-4,
        lr_mhc=2.5e-5,
        unlikelihood_alpha=0.25,
        opencode_api_url=os.environ.get('OPENCODE_API_URL', None),
        opencode_api_key=os.environ.get('OPENCODE_API_KEY', None),
        opencode_model=os.environ.get('OPENCODE_MODEL', 'opencode/claude-sonnet-4-5'),
        save_dir='/content/sparkx_v03_si_srsi_engram',
        push_to_hub=PUSH_CONSOLIDATED_TO_HUB and bool(HF_TOKEN),
        hub_repo_id=HF_REPO_ID,
        token=HF_TOKEN,
    )